In [ ]:
import pyodbc
import pandas.io.sql as psql
import numpy as np
from sqlalchemy import create_engine
import urllib
import pandas as pd

In [ ]:
### Connection ###

conn = pyodbc.connect(r"DRIVER={SQL Server};"             
               r"Server=Hvac-phdb-db.apps.carrier.com;"
               r"Database=CarrierAnalytics;"
               r"UID=cads;"
               r"PWD=zs823j1903;"                      
                      )
cursor = conn.cursor()

#Monthly Data
year_warranty=pd.read_sql_query("""Select ClaimNumber,Model,Serial,ReplPartNumCompress, ReplacementPartDescription, Plant,EquipmentState,ClaimDate,PaidDate,ManufactureDate,DateFailed,DateRepaired,StartupDate,ShipDate,ClaimType,TimeType,LaborAmountApproved,LaborHours,ServicePerformed,WarrantyType,WtyTypeDescription,GLTotal,GLStandardExt,TotalClaimAmountAppr,GLStandardTot,GLStandard1stYr,GLOptional,GLConcessions,GLMargin,GLProductTrouble,clmqty,qty,PRODUCT_WLC_STD,SalesCat,WarrCName,BusinessUnit,DefectCode,CustomerCompanyName,ReferenceNumber FROM CarrierAnalytics.dbo.[WarrantyGL2YearHistory] Where Plant In ('CMXB','Charlotte') """, conn)
year_warranty.PaidDate=pd.to_datetime(year_warranty.PaidDate)
year_warranty.DateFailed=pd.to_datetime(year_warranty.DateFailed)
print(year_warranty.DateFailed.min(),year_warranty.DateFailed.max())
print(year_warranty.PaidDate.min(),year_warranty.PaidDate.max())
## Drop Duplicates
old=year_warranty.drop_duplicates(['Serial','ClaimNumber','ServicePerformed','GLTotal','TotalClaimAmountAppr','GLStandardTot','GLStandard1stYr','GLOptional','GLProductTrouble'],keep='first')  #How to put this into SQL code???
old['Beyond1stYr']=old['GLStandardExt']

#Monthly Data
monthly_warranty=pd.read_sql_query("""Select ClaimNumber,Model,Serial,ReplPartNumCompress, ReplacementPartDescription, Plant,EquipmentState,ClaimDate,PaidDate,ManufactureDate,DateFailed,DateRepaired,StartupDate,ShipDate,ClaimType,TimeType,LaborAmountApproved,LaborHours,ServicePerformed,WarrantyType,WtyTypeDescription,GLTotal,GLStandardExt,TotalClaimAmountAppr,GLStandardTot,GLStandard1stYr,GLOptional,GLConcessions,GLMargin,GLProductTrouble,clmqty,qty,PRODUCT_WLC_STD,SalesCat,WarrCName,BusinessUnit,DefectCode,CustomerCompanyName,ReferenceNumber FROM CarrierAnalytics.dbo.[WarrantyGLNewMonthHistory] Where Plant In ('CMXB','Charlotte') """, conn)
monthly_warranty.PaidDate=pd.to_datetime(monthly_warranty.PaidDate)
monthly_warranty.DateFailed=pd.to_datetime(monthly_warranty.DateFailed)
print(monthly_warranty.DateFailed.min(),monthly_warranty.DateFailed.max())
print(monthly_warranty.PaidDate.min(),monthly_warranty.PaidDate.max())
## Drop Duplicates
monthly_warranty=monthly_warranty.drop_duplicates(['Serial','ClaimNumber','ServicePerformed','GLTotal','TotalClaimAmountAppr','GLStandardTot','GLStandard1stYr','GLOptional','GLProductTrouble'],keep='first')  #How to put this into SQL code???
monthly_warranty['Beyond1stYr']=monthly_warranty['GLStandardExt']


In [ ]:
old = old[(old.PaidDate<monthly_warranty.PaidDate.min())]
hist_full = pd.concat([old,monthly_warranty])
hist_full = hist_full[list(monthly_warranty.columns)]
hist_full

In [ ]:
duplicado_full=hist_full[hist_full['Serial'].duplicated(keep=False)].reset_index(drop=True)
duplicado_small = duplicado_full[(duplicado_full['GLStandard1stYr']!=0)&(duplicado_full['ClaimType']=='Standard')&(duplicado_full['WarrantyType']!='1S')&(duplicado_full['TimeType'].isin(['FFR','DOA',90]))]
duplicado_big = duplicado_full[(duplicado_full['GLStandard1stYr']==0)|(duplicado_full['ClaimType']!='Standard')|(duplicado_full['WarrantyType']=='1S')|(~duplicado_full['TimeType'].isin(['FFR','DOA',90]))]
no_duplicado=hist_full[~hist_full['Serial'].duplicated(keep=False)].reset_index(drop=True)
start=0
for plants in duplicado_small.Plant.unique():
    print(plants)
    start=start+1
    duplicado=duplicado_small[(duplicado_small.Plant==plants)]
    print('Len Duplo',len(duplicado),'\n\n\n')
    for i in range (duplicado['Serial'].nunique()):
        print(duplicado['Serial'].nunique())
        duplo=duplicado[duplicado['Serial']==duplicado['Serial'].unique()[i]]
        duplo=duplo.sort_values(['PaidDate'], ascending=True).reset_index(drop=True)
        duplo=duplo.sort_values(['Serial','DateFailed','clmqty'], ascending=True).reset_index(drop=True)
        #print(duplicado['Serial'].unique()[i])     
        X=[]
        a,b,c=0,0,0
        for j in range(len(duplo)):
            #print(j)
            if (duplo['clmqty'][j]==1):
                a=a+1
            if (duplo['clmqty'][j]==0):
                b=b+1
            if (duplo['clmqty'][j]==-1):
                c=c+1
            if j ==0:
                X.append('yes')
            else:
                dias=abs((duplo['DateFailed'][j-1]-duplo['DateFailed'][j]).days)
                if dias >= 30:
                    X.append('yes')
                else:
                    if duplo['clmqty'][j-1]!=duplo['clmqty'][j]:
                        if (duplo['clmqty'][j]==1)&(a<=1):
                            X.append('yes')
                        else:
                            if (duplo['clmqty'][j]==0)&(b<=1):
                                X.append('yes')
                            else:
                                if (duplo['clmqty'][j]==-1)&(c<=1):
                                    X.append('yes')
                                else:
                                    X.append('no')
                    else:
                        X.append('no')
        print(len(duplo),len(X))
        duplo['15.- Comm Count']=X
        if i >0:
            duplicated=pd.concat([duplicated, duplo])
        else:
            duplicated=duplo
    #X=pd.DataFrame(X)
    #X.columns=['15.- Comm Count']
    duplicado=duplicated#pd.concat([duplicated.reset_index(drop=True), pd.DataFrame(X).reset_index(drop=True)], axis=1)
    if start>1:
        duplicado_concat=pd.concat([duplicado_concat,duplicado])
    else:
        duplicado_concat=duplicado
no_duplicado.loc[no_duplicado['GLStandard1stYr']!=0, '15.- Comm Count']='yes'
duplicado_big['15.- Comm Count']='no'
df=pd.concat([duplicado_concat,duplicado_big, no_duplicado])

hist_full=df

##Does the 30 day rule applies for Residential Also. Well I don't Know???
hist_full.reset_index(drop=True, inplace=True)
hist_full['Quantity']=0
hist_full['Quantity w/ 1S']=0
hist_full.loc[(hist_full['ClaimType']=='Standard')&(hist_full['WarrantyType']!='1S')& (hist_full['TimeType'].isin(['FFR','DOA',90]))&(hist_full['15.- Comm Count']=='yes'), 'Quantity'] = hist_full['clmqty']
#hist_full.loc[(hist_full['ClaimType']=='Standard')& (hist_full['TimeType'].isin(['FFR','DOA',90]))&(hist_full['15.- Comm Count']=='yes'), 'Quantity w/ 1S'] = hist_full['clmqty']

In [ ]:
print(len(hist_full),hist_full.PaidDate.min(),hist_full.PaidDate.max())

## Drop Duplicates
new_data=hist_full
print(len(new_data))

new_data['duplo']=new_data[['Serial','ClaimNumber','ServicePerformed','GLTotal','TotalClaimAmountAppr','GLStandardTot','GLStandard1stYr','GLOptional','GLProductTrouble']].astype('str').agg(' '.join, axis=1)
#last['duplo']=last[['Serial','ClaimNumber','ServicePerformed','GLTotal','TotalClaimAmountAppr','GLStandardTot','GLStandard1stYr','GLOptional','GLProductTrouble']].astype('str').agg(' '.join, axis=1)

#new_data=new_data[~new_data['duplo'].isin(last.duplo.unique())]
print(len(new_data))
print('New Data Paid Dates ',new_data.PaidDate.min(),new_data.PaidDate.max())
new_data

In [ ]:
# ============================================================
# GEMMA AI CLASSIFICATION - GROUPED BY SERIAL WITHIN 30 DAYS
# ============================================================

import time
import pandas as pd
import importlib
import best_gemma_classifier as bgc
importlib.reload(bgc)

from best_gema_config import (
    PRIMARY_COMPONENTS,
    SECONDARY_COMPONENTS,
    FAILURE_REASONS,
    map_secondary_group,
    normalize_secondary,
)

MODEL = "gemma4:12B"




SAMPLE_SIZE = 2
RANDOM_STATE = 42
GROUPING_DAYS = 30
OUTPUT_XLSX = f"REPORT_serial_group_sample_{SAMPLE_SIZE}.xlsx"
OUTPUT_CSV = f"RESULTS_serial_group_sample_{SAMPLE_SIZE}.csv"
SERIAL_COL = "Serial"
SERVICE_COL = "ServicePerformed"
PART_COL = "ReplacementPartDescription"

GROUP_DATE_COL = "ClaimDate"   # use ClaimDate for failure grouping
FALLBACK_DATE_COL = "PaidDate"  # fallback if ClaimDate is missing


def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_key(value):
    return (
        str(value)
        .strip()
        .upper()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace("/", "")
    )


def force_match(value, taxonomy_list):
    cleaned = normalize_key(value)

    for item in taxonomy_list:
        if cleaned == normalize_key(item):
            return item

    return value


def normalize_to_taxonomy(value, taxonomy_list):
    if pd.isna(value):
        return ""

    if taxonomy_list == SECONDARY_COMPONENTS:
        grouped = map_secondary_group(normalize_secondary(value))
        return force_match(grouped, SECONDARY_COMPONENTS)

    return force_match(value, taxonomy_list)


def build_serial_groups(df):
    df = df.copy()

    df[GROUP_DATE_COL] = pd.to_datetime(df[GROUP_DATE_COL], errors="coerce")
    df[FALLBACK_DATE_COL] = pd.to_datetime(df[FALLBACK_DATE_COL], errors="coerce")

    df["ai_group_date"] = df[GROUP_DATE_COL]
    df.loc[df["ai_group_date"].isna(), "ai_group_date"] = df[FALLBACK_DATE_COL]

    df[SERIAL_COL] = df[SERIAL_COL].astype(str).str.strip()

    df = df[
        df[SERIAL_COL].notna()
        & (df[SERIAL_COL].astype(str).str.strip() != "")
        & df[SERVICE_COL].notna()
        & (df[SERVICE_COL].astype(str).str.strip() != "")
    ].copy()

    df = df.sort_values(
        [SERIAL_COL, "ai_group_date", "ClaimNumber"],
        ascending=True
    ).reset_index(drop=True)

    group_ids = []

    current_group_id = 0
    previous_serial = None
    previous_date = None

    for _, row in df.iterrows():
        serial = row[SERIAL_COL]
        current_date = row["ai_group_date"]

        start_new_group = False

        if serial != previous_serial:
            start_new_group = True
        elif pd.isna(current_date) or pd.isna(previous_date):
            start_new_group = True
        else:
            days_diff = abs((current_date - previous_date).days)

            if days_diff >= GROUPING_DAYS:
                start_new_group = True

        if start_new_group:
            current_group_id += 1

        group_ids.append(current_group_id)

        previous_serial = serial
        previous_date = current_date

    df["AIGroupID"] = group_ids

    return df


def combine_group_text(group):
    service_lines = []

    for _, row in group.iterrows():
        claim = clean_text(row.get("ClaimNumber", ""))
        claim_date = clean_text(row.get("ClaimDate", ""))
        part = clean_text(row.get(PART_COL, ""))
        service = clean_text(row.get(SERVICE_COL, ""))

        line = (
            f"ClaimNumber: {claim}\n"
            f"ClaimDate: {claim_date}\n"
            f"ReplacementPartDescription: {part}\n"
            f"ServicePerformed: {service}"
        )

        service_lines.append(line)

    return "\n\n--- RELATED CLAIM IN SAME SERIAL GROUP ---\n\n".join(service_lines)


def combine_parts(group):
    parts = (
        group[PART_COL]
        .dropna()
        .astype(str)
        .str.strip()
    )

    parts = [p for p in parts if p]

    if not parts:
        return "Not Provided"

    return " | ".join(sorted(set(parts)))


# ------------------------------------------------------------
# 1. Build grouped dataframe
# ------------------------------------------------------------

ai_row_level = build_serial_groups(new_data)

grouped_rows = []

for group_id, group in ai_row_level.groupby("AIGroupID"):
    serial = group[SERIAL_COL].iloc[0]

    grouped_rows.append({
        "AIGroupID": group_id,
        "Serial": serial,
        "Plant": " | ".join(sorted(set(group["Plant"].dropna().astype(str)))) if "Plant" in group.columns else "",
        "ClaimNumbers": " | ".join(group["ClaimNumber"].dropna().astype(str).unique()),
        "GroupClaimCount": len(group),
        "GroupStartDate": group["ai_group_date"].min(),
        "GroupEndDate": group["ai_group_date"].max(),
        "CombinedReplacementParts": combine_parts(group),
        "CombinedServiceText": combine_group_text(group),
        "TotalClaimAmountAppr": group["TotalClaimAmountAppr"].sum() if "TotalClaimAmountAppr" in group.columns else None,
        "GLTotal": group["GLTotal"].sum() if "GLTotal" in group.columns else None,
    })

ai_grouped = pd.DataFrame(grouped_rows)

print(f"Total original rows available for AI: {len(ai_row_level)}")
print(f"Total serial/date groups available for AI: {len(ai_grouped)}")

# ------------------------------------------------------------
# 2. Take  group sample
# ------------------------------------------------------------

sample_groups = ai_grouped.sample(
    n=min(SAMPLE_SIZE, len(ai_grouped)),
    random_state=RANDOM_STATE
).copy()

print(f"Groups selected for Gemma sample: {len(sample_groups)}")

# ------------------------------------------------------------
# 3. Create prediction columns
# ------------------------------------------------------------

sample_groups["pred_primary_gemma"] = ""
sample_groups["pred_secondary_gemma"] = ""
sample_groups["pred_failure_gemma"] = ""
sample_groups["needs_review"] = False
sample_groups["review_reason"] = ""

# ------------------------------------------------------------
# 4. Run Gemma classification
# ------------------------------------------------------------

start_time = time.perf_counter()

for counter, (idx, row) in enumerate(sample_groups.iterrows(), start=1):
    print(f"[{counter}/{len(sample_groups)}] Running Gemma for AIGroupID {row['AIGroupID']}")
    service_text = clean_text(row["CombinedServiceText"])
    replacement_part = clean_text(row["CombinedReplacementParts"]) or "Not Provided"
    result = bgc.classify(
        text=service_text,
        replacement_part_description=replacement_part,
        model=MODEL,
    )

    if (
        not result
        or not result.get("primary_component")
        or not result.get("secondary_component")
        or not result.get("failure_reason")
    ):
        print(f"   ! Empty result for AIGroupID {row['AIGroupID']}. Retrying with llama3.")
        result = bgc.classify(
            text=service_text,
            replacement_part_description=replacement_part,
            model="llama3",
        )

    pred_primary = result.get("primary_component", "")
    pred_secondary = result.get("secondary_component", "")
    pred_failure = result.get("failure_reason", "")

    sample_groups.at[idx, "pred_primary_gemma"] = pred_primary
    sample_groups.at[idx, "pred_secondary_gemma"] = pred_secondary
    sample_groups.at[idx, "pred_failure_gemma"] = pred_failure


    review_reasons = []

    if pred_primary == "Ambiguous" or pred_secondary == "Ambiguous" or pred_failure == "Ambiguous":
        review_reasons.append("Ambiguous prediction")


    if len(service_text) < 40:
        review_reasons.append("Very short service text")

    if review_reasons:
        sample_groups.at[idx, "needs_review"] = True
        sample_groups.at[idx, "review_reason"] = " | ".join(review_reasons)

elapsed = (time.perf_counter() - start_time) / 60

# ------------------------------------------------------------
# 5. Merge predictions back to row-level data
# ------------------------------------------------------------

prediction_cols = [
    "AIGroupID",
    "pred_primary_gemma",
    "pred_secondary_gemma",
    "pred_failure_gemma",
    "needs_review",
    "review_reason",
]

ai_row_level_sample = ai_row_level.merge(
    sample_groups[prediction_cols],
    on="AIGroupID",
    how="inner"
)

# ------------------------------------------------------------
# 6. Build summary
# ------------------------------------------------------------

summary = pd.DataFrame([
    {"Metric": "Original rows in new_data", "Value": len(new_data)},
    {"Metric": "Rows eligible for AI", "Value": len(ai_row_level)},
    {"Metric": "Serial/date groups available", "Value": len(ai_grouped)},
    {"Metric": "Groups classified in sample", "Value": len(sample_groups)},
    {"Metric": "Row-level records represented in sample", "Value": len(ai_row_level_sample)},
    {"Metric": "Needs review groups", "Value": int(sample_groups["needs_review"].sum())},
    {"Metric": "Needs review rate", "Value": sample_groups["needs_review"].mean()},
    {"Metric": "Runtime minutes", "Value": elapsed},
])

primary_dist = (
    sample_groups["pred_primary_gemma"]
    .value_counts()
    .reset_index()
)
primary_dist.columns = ["pred_primary_gemma", "Count"]

secondary_dist = (
    sample_groups["pred_secondary_gemma"]
    .value_counts()
    .reset_index()
)
secondary_dist.columns = ["pred_secondary_gemma", "Count"]

failure_dist = (
    sample_groups["pred_failure_gemma"]
    .value_counts()
    .reset_index()
)
failure_dist.columns = ["pred_failure_gemma", "Count"]

needs_review = sample_groups[sample_groups["needs_review"]].copy()

# ------------------------------------------------------------
# 7. Export
# ------------------------------------------------------------

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Summary", index=False)
    sample_groups.to_excel(writer, sheet_name="ClassifiedGroups", index=False)
    ai_row_level_sample.to_excel(writer, sheet_name="ClassifiedRows", index=False)
    needs_review.to_excel(writer, sheet_name="NeedsReview", index=False)
    primary_dist.to_excel(writer, sheet_name="PrimaryDistribution", index=False)
    secondary_dist.to_excel(writer, sheet_name="SecondaryDistribution", index=False)
    failure_dist.to_excel(writer, sheet_name="FailureDistribution", index=False)

sample_groups.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\nGemma AI classification completed.")
print(f"Runtime: {elapsed:.2f} minutes")
print(f"Excel created: {OUTPUT_XLSX}")